In [10]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch


In [11]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA GeForce GTX 1650


In [12]:
start = int(input("Enter Start Year: "))
end = int(input("Enter Ending Year: "))


In [14]:
# df = pd.read_parquet(f"D:\LPA_MTech_Project\Enriched_Datasets\SupremeCourt_Combined_{start}_{end}_enriched.parquet")

# df = df[["text", "verdict_label"]].dropna()
# df = df.rename(columns={"verdict_label": "label"})

# dataset = Dataset.from_pandas(df).train_test_split(test_size=0.15)
import datasets  # needed for Value()
df = pd.read_parquet(
    f"D:/LPA_MTech_Project/Enriched_Datasets/SupremeCourt_Combined_{start}_{end}_enriched.parquet"
)

# Keep required columns
df = df[["text", "verdict_label"]].dropna()

# Rename label column
df = df.rename(columns={"verdict_label": "label"})

# Ensure labels are integer class indices (0/1)
df["label"] = df["label"].astype(int)

# Reset index to avoid extra index column in HF dataset
df.reset_index(drop=True, inplace=True)

# Convert to HF dataset
dataset = datasets.Dataset.from_pandas(df)

# FIXES LABEL SHAPE ISSUE
dataset = dataset.cast_column("label", datasets.Value("int64"))

# Train-test split
dataset = dataset.train_test_split(test_size=0.15, seed=42)


Casting the dataset: 100%|██████████| 4061/4061 [00:00<00:00, 23187.87 examples/s]


In [ ]:
def clean_text(t):
    t = str(t)
    t = t.replace("\n", " ").replace("\t", " ")
    t = " ".join(t.split())   # remove duplicate spaces
    return t

df["text"] = df["text"].apply(clean_text)


In [ ]:
df["text"] = df["text"].str[:5000]   # cut raw text before tokenization


In [ ]:
model_name = "nlpaueb/legal-bert-base-uncased"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized = dataset.map(tokenize, batched=True, batch_size=16)
tokenized = tokenized.remove_columns(["text"])
tokenized.set_format("torch")


Map: 100%|██████████| 610/610 [00:10<00:00, 57.44 examples/s]


In [ ]:
# training_args = TrainingArguments(
#     output_dir="legalbert_verdict",
#     num_train_epochs=4,

#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=8,

#     learning_rate=2e-5,
#     warmup_steps=200,
#     weight_decay=0.01,

#     fp16=True,                        # GPU safe
#     logging_steps=50,

#     eval_strategy="epoch",            # transformers >=4.57 uses eval_strategy
#     save_strategy="epoch",

#     dataloader_num_workers=0,         # Windows fix
#     gradient_checkpointing=True,      # reduces VRAM use
# )

training_args = TrainingArguments(
    output_dir="legalbert_verdict",
    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,

    fp16=True,
    logging_steps=50,

    eval_strategy="epoch",
    save_strategy="epoch",

    dataloader_num_workers=0,   # Windows safe
    gradient_checkpointing=False,   # MUCH faster
)


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
)


In [19]:
trainer.train()


  3%|▎         | 50/1724 [06:51<3:45:30,  8.08s/it]

{'loss': 5.5022, 'grad_norm': 49.17626190185547, 'learning_rate': 4.600000000000001e-06, 'epoch': 0.12}


  6%|▌         | 100/1724 [13:37<3:39:08,  8.10s/it]

{'loss': 4.5492, 'grad_norm': 21.504465103149414, 'learning_rate': 9.600000000000001e-06, 'epoch': 0.23}


  9%|▊         | 150/1724 [20:21<3:32:19,  8.09s/it]

...

{'loss': 2.62, 'grad_norm': 131.74461364746094, 'learning_rate': 1.2847769028871394e-05, 'epoch': 1.74}


 46%|████▋     | 800/1724 [1:50:46<2:04:26,  8.08s/it]

{'loss': 2.7504, 'grad_norm': 48.517539978027344, 'learning_rate': 1.2191601049868766e-05, 'epoch': 1.85}


 48%|████▊     | 830/1724 [1:54:49<2:01:46,  8.17s/it]

KeyboardInterrupt: 

In [ ]:
results = trainer.evaluate()
print(results)
